# Attempt Day 1

based on session notes, the defined classes and excution happy paths. are the core outer requirements to build

In [1]:
class ParkingSpot:
    def __init__(self):
        self.occupant = None

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is not None: 
            return False
        else: #is None, hence self.occupant can be set to vehicle.
            self.occupant = vehicle
            return True
    
    # check unpark for later.
    def release(self):
        if self.occupant is None: 
            return False
        else: 
            self.occupant = None
            return True
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self):
        pass

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self):
        pass

class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = [ParkingSpot() for _ in range(self.size)]
        self.size = size
        self.filled = 0

    def full(self):
        if self.size == self.filled:
            return True
        else:
            return False

    # TODO: Move controller pattern from ParkingFloor to ParkingLot
    # TODO:
    def park(self, vehicle):
        if not self.full():
            #naive parkingspot implementation:
            for parking_spot in self.parking_grid:
                if parking_spot.assign(vehicle):
                    break
            self.filled += 1
        
    def unpark(self, vehicle):
        for parking_spot in self.parking_grid:
            if parking_spot.get_occupant() == vehicle and parking_spot.release(vehicle):
                self.filled -= 1
    
class ParkingLot:
    def __init__(self):
        pass
    def park(self, vehicle):
        pass
    def unpark(self,vehicle):
        pass


### Critique After Attempt 1 (Hard, no spoilers)

Reference: `../2026-05-27-session.md` sections **2A, 2E, 4A, 5A**.

- `High:` You coded classes before locking invariants/state machine. This repeats the workflow skip called out in notes (`2B`, `5A`): you are still describing behavior, not proving correctness constraints.
- `High:` Ownership is split wrong: `ParkingFloor.park/unpark` still mutates core lifecycle while your own notes converged that lot-level transitions must be owned by `ParkingLot` (`4C`: mutation closure).
- `High:` `Ticket` is structurally empty for exit semantics. Notes explicitly challenged this (`2E`, `2G`): unpark keyed by session handle is impossible to guarantee with current model.
- `Medium:` `ParkingSpot` has occupancy but no compatibility state, despite notes saying compatibility is a first-class requirement (`1B`, `2A`, `2E`).
- `Medium:` There is no invariant guard for double-booking across the system boundary; local `assign` checks are not enough as a design claim (`5A`, `5C`).

Verdict: this attempt is implementation-first and still under-modeled at the LLD level.


## Attempt 1 Continued but refactoring

1. It seems that ParkingLot is an aggregate root since it owns one to many ParkingFloors and ParkingFloors own ParkingSpots.
2. ParkingSpot should track both compatibility and occupancy, not only which vehicle is currently there.
3. Ticket likely needs more than entry time. At minimum, think about whether the system can unpark correctly if the ticket does not identify the parked vehicle or spot.
4. ParkingLot and ParkingFloor should just be collections instead of state tracking from previous
5. Move parking controller logic to ParkingLot as the core aggregate

Run with naive search and park unpark implemenation for now.

In [ ]:
class ParkingSpot:
    def __init__(self):
        self.occupant = None

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is not None: 
            return False
        else: #is None, hence self.occupant can be set to vehicle.
            self.occupant = vehicle
            return True
    
    # check unpark for later.
    def release(self):
        if self.occupant is None: 
            return False
        else: 
            self.occupant = None
            return True
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type = None, ticket = None):
        self.id = id
        self.type = type
        self.ticket = ticket
        

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self, id = None, entry_time = None):
        self.id = id
        self.entry_time = entry_time

class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = [ParkingSpot() for _ in range(self.size)]
        self.size = size
        self.filled = 0

    def full(self):
        if self.size == self.filled:
            return True
        else:
            return False

    # TODO: Move controller pattern from ParkingFloor to ParkingLot
    # TODO:
    def park(self, vehicle):
        if not self.full():
            #naive parkingspot implementation:
            for parking_spot in self.parking_grid:
                if parking_spot.assign(vehicle):
                    break
            self.filled += 1
        
    def unpark(self, vehicle):
        for parking_spot in self.parking_grid:
            if parking_spot.get_occupant() == vehicle and parking_spot.release(vehicle):
                self.filled -= 1
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size)]
    def park(self, vehicle):
        parked = False
        for parking_floor in self.parking_floors:
            if parking_floor.park(vehicle):
                parked = True
                break
        return parked

    def unpark(self,vehicle):
        unparked = False
        for parking_floor in self.parking_floors:
            if parking_floor.unpark(vehicle):
                unparked = True
                break
        return unparked


This is a great question because it gets to the heart of **API design, DDD aggregates, identity, and state transitions**.

Let's analyze it from first principles.

---

# 1. The core operation is a state transition

The parking lot state can be modeled as:

[
ParkingLotState = { Spots, Vehicles, Tickets }
]

The two transitions are:

```
park(vehicle)   : State → State
unpark(ticket)  : State → State
```

The question is:

> What information is the minimal, stable, authoritative identity needed to perform each transition?

---

# 2. Why park takes a Vehicle

## Intent

The user's intent is:

> "I have a vehicle. Please find a place for it."

Before parking, the vehicle has no relationship with the parking lot.

The system needs information from the vehicle:

```python
Vehicle {
    id
    type  # car, motorcycle, truck
}
```

The algorithm is:

```
park(vehicle):
    spot = find_available_spot(vehicle.type)
    occupy(spot, vehicle)
    ticket = issue_ticket(vehicle, spot, entry_time)
    return ticket
```

The **vehicle determines the constraints**.

The parking lot is solving a search problem:

```
Vehicle type
      |
      v
Find compatible spot
      |
      v
Create parking relation
```

So:

```
park(vehicle)
```

matches the direction of causality.

---

# 3. Why unpark takes a Ticket

Now the situation is reversed.

After parking, the authoritative relationship is:

```
Ticket
  |
  +-- vehicle_id
  |
  +-- spot_id
  |
  +-- entry_time
```

The ticket is a **proof of an existing parking transaction**.

Unparking asks:

> "Which parking relationship should I terminate?"

You already know the answer from the ticket.

```
unpark(ticket):
    spot = ticket.spot_id
    release(spot)
    calculate_fee(ticket)
    close(ticket)
```

This is O(1).

---

# 4. Why not unpark(vehicle)?

Imagine:

```python
unpark(vehicle)
```

The system has to ask:

```
Where is this vehicle parked?
```

So you need a lookup:

```
vehicle_id → active_ticket → spot
```

which means either:

* maintain another index
* scan all tickets
* handle duplicate plates, errors, etc.

The vehicle is not the primary identity of the parking event.

---

# 5. The deeper DDD principle: identify the aggregate by the lifecycle event

A useful rule:

> **Commands should enter an aggregate using the identity that uniquely represents the current lifecycle stage.**

Lifecycle:

```
Before parking:

Vehicle
   |
   | park()
   v

ParkingSession (Ticket)
   |
   | unpark()
   v

Completed
```

The "root identity" changes as the state machine evolves.

---

# 6. Another way to see it: create vs destroy

Parking creates a relationship:

```
Vehicle + EmptySpot
          |
          v
    ParkingSession
```

Therefore the input is the thing being introduced:

```
park(vehicle)
```

---

Unparking destroys the relationship:

```
ParkingSession
      |
      v
Vehicle + EmptySpot
```

Therefore the input is the relationship identifier:

```
unpark(ticket)
```

---

# 7. General design pattern

This appears everywhere.

## Hotel

Check-in:

```python
check_in(guest)
```

creates:

```
Reservation / Stay
```

Check-out:

```python
check_out(stay_id)
```

---

## Shopping

Create order:

```python
place_order(cart)
```

Later:

```python
cancel_order(order_id)
```

---

## Banking

Open account:

```python
open_account(customer)
```

Later:

```python
close_account(account_id)
```

---

# Compression table

| Operation type                | Input identity                   | Principle                                               |
| ----------------------------- | -------------------------------- | ------------------------------------------------------- |
| Create relationship           | External entity                  | "What is entering the system?"                          |
| Modify/terminate relationship | Relationship ID / transaction ID | "Which existing state transition is being manipulated?" |
| Query state                   | Most selective identifier        | "What lets us locate state efficiently?"                |

---

So in DDD terms, **`park(vehicle)` is a command that creates a new aggregate instance (a Parking Session/Ticket), while `unpark(ticket)` is a command against an existing aggregate instance.**

The deepest principle is:

> **Commands should be expressed in terms of the entity that owns the information necessary to make the state transition valid.**

For parking, that is the **vehicle's constraints**. For unparking, that is the **ticket's identity and history**.


What you are asking for is essentially the **design derivation process**: *how do I discover the right command boundary and identity?* This is one of the central skills in DDD and low-level design.

The chain of thought is not "what objects exist?" but rather **what state transition am I trying to control?**

---

## Step 1: Ask: What is the state machine?

Do not start with classes.

Start with:

```
Empty Spot + Vehicle
        |
        | park()
        v
Occupied Spot + Parking Session
        |
        | unpark()
        v
Empty Spot + Vehicle exits
```

Now you see there is a new thing created:

```
Parking Session (Ticket)
```

This is the thing whose lifecycle matters.

---

## Step 2: Ask: What information is missing before the transition?

For `park`:

Current state:

```
ParkingLot
  spots: empty
```

Input:

```
Vehicle(type=Car)
```

Question:

> What do I need to decide whether this transition is valid?

You need:

* vehicle type
* vehicle identity (for tracking)

You **do not have a ticket yet**.

So the command must be:

```python
park(vehicle)
```

---

## Step 3: Ask: What identity is created by the transition?

After parking:

```
Ticket {
    ticket_id
    vehicle_id
    spot_id
    entry_time
}
```

The parking operation creates a **new relationship**:

```
Vehicle <---- Ticket ----> Spot
```

The ticket becomes the canonical identity of this parking session.

---

## Step 4: Ask: What does the reverse transition need to reference?

For `unpark`:

The intent is:

> "End a specific parking session."

Now ask:

**What uniquely identifies a parking session?**

Candidates:

### Vehicle

```
vehicle_id → active ticket → spot
```

Requires an additional lookup.

Also has ambiguity:

```
Vehicle A parked Monday
Vehicle A parked Tuesday
```

Which parking event?

---

### Spot

```
unpark(spot_id)
```

Question:

> Who is authorized to release this spot?

Anyone who knows the spot number can do it.

Bad domain boundary.

---

### Ticket

```
ticket_id → spot, vehicle, entry time
```

The ticket directly points to the exact relationship.

Therefore:

```python
unpark(ticket)
```

is the natural command.

---

## Step 5: Ask: What is the aggregate's current identity?

This is a very DDD way to think.

The "identity of the thing under control" changes over time.

Before parking:

```
Vehicle
```

After parking:

```
ParkingSession
```

So:

```
park(vehicle)
       |
       v
ParkingSession(ticket)
       |
unpark(ticket)
```

---

## The general design algorithm

Whenever designing an API, run this checklist:

### 1. What state transition is occurring?

```
Before ---- command ----> After
```

---

### 2. What entity is being created or destroyed?

```
A + B → Relationship
```

or

```
Relationship → A + B
```

---

### 3. What information is necessary to validate the transition?

Examples:

* vehicle type?
* account balance?
* reservation date?

---

### 4. After the transition, what becomes the authoritative identity?

Examples:

```
Order ID
Ticket ID
Reservation ID
Session ID
```

---

### 5. For future commands, use the identity of the current lifecycle stage.

---

## Compression rule

A very powerful heuristic:

| Operation                        | Usually takes                         |
| -------------------------------- | ------------------------------------- |
| Create a relationship            | The participating entities            |
| Change an existing relationship  | The relationship's ID                 |
| Destroy an existing relationship | The relationship's ID                 |
| Query                            | The smallest authoritative identifier |

Examples:

```
placeOrder(cart)         → Order
cancelOrder(orderId)

bookFlight(customer)     → Reservation
cancelFlight(reservationId)

login(credentials)       → Session
logout(sessionId)

park(vehicle)            → Ticket
unpark(ticket)
```

---

The biggest mindset shift is:

**Don't design around nouns ("I have Vehicle and Spot, so methods should take Vehicle"). Design around transitions in a state machine. The parameter is the information needed to move from one valid state to another, and over time the authoritative identity often changes.**

This is the same mental model used in DDD aggregates, event sourcing, finite state machines, and even control theory (commands acting on a system state).


## Interview Tips:

This pattern has several formal interpretations depending on the field. The most useful one for software design is to think of it as an **associative entity (relationship entity)** with its own **identity and lifecycle**.

Let's build the vocabulary.

---

# 1. Database / ER Modeling: Associative Entity

In an ER diagram:

Before parking:

```
Vehicle -------- Spot
```

The relationship "is parked in" is not just a boolean relation because it has attributes:

```
Vehicle
    |
    |
ParkingSession (Ticket)
    |
    |
ParkingSpot
```

Formally:

```
Vehicle ---< ParkingSession >--- Spot
```

The middle object is an **associative entity** (also called a **junction entity**, **link entity**, or **bridge entity**).

It represents the relationship itself as a first-class object.

### Why does it exist?

Because the relation has its own data:

```
ParkingSession {
    id
    vehicle_id
    spot_id
    entry_time
    exit_time
    fee
}
```

A plain edge:

```
Vehicle → Spot
```

cannot naturally hold all this state.

---

# 2. Domain-Driven Design: Aggregate / Domain Entity

In DDD terms:

```
Vehicle        ParkingSpot
    \              /
     \            /
      ParkingSession
```

The parking session is a **domain entity** with:

* identity (`ticket_id`)
* lifecycle
* invariants

Examples:

```
entry_time < exit_time
fee must be paid before close
only one active session per spot
```

So after creation, the command target changes:

```
park(vehicle)
        |
        v
ParkingSession(ticket)
        |
        v
unpark(ticket)
```

---

# 3. Relational Algebra: A Relation with Attributes

Mathematically:

Initially you might think:

[
ParkedIn \subseteq Vehicle \times Spot
]

a simple binary relation.

But reality is:

[
ParkingSession \subseteq Vehicle \times Spot \times Time \times Money \times Status
]

It becomes an **n-ary relation**.

The ticket is the identity of one tuple in this relation.

---

# 4. Category Theory: A Morphism Object / Span

Very abstractly:

```
Vehicle  <- ParkingSession -> Spot
```

is a **span**.

A span is an object that relates two objects through two morphisms.

```
          ParkingSession
          /             \
         /               \
   Vehicle             Spot
```

This appears in databases, graph transformations, and applied category theory.

---

# 5. Graph Theory: Edge Promotion (Reification)

A graph edge:

```
Vehicle ----- Spot
```

is turned into a node:

```
Vehicle --- ParkingSession --- Spot
```

This process is called:

* **edge reification**
* **relationship reification**

because the relationship becomes a thing.

---

# The core design pattern

The general pattern is:

```
A + B
 |
 | create relation
 v
Relationship(A, B)
```

where the relationship has:

* its own identity
* state over time
* business rules

---

# Common software examples

| Domain         | Relationship entity   | Before              | After             |
| -------------- | --------------------- | ------------------- | ----------------- |
| Parking        | ParkingSession/Ticket | Vehicle + Spot      | Ticket            |
| Hotel          | Stay / Reservation    | Guest + Room        | Reservation ID    |
| Airline        | Booking               | Passenger + Flight  | Booking ID        |
| Shopping       | Order                 | Customer + Products | Order ID          |
| Banking        | Account               | Customer + Bank     | Account ID        |
| Employment     | EmploymentContract    | Person + Company    | Contract ID       |
| Education      | Enrollment            | Student + Course    | Enrollment ID     |
| Social media   | Friendship            | User + User         | Friendship object |
| Git            | Pull Request          | Branch + Repository | PR ID             |
| Messaging      | Conversation          | Users               | Conversation ID   |
| Authentication | Session               | User + Device       | Session ID        |
| Ride sharing   | Trip                  | Rider + Driver      | Trip ID           |
| Delivery       | Shipment              | Order + Carrier     | Tracking ID       |

---

# How to spot this pattern in design interviews

Ask these questions:

### 1. Is the relationship long-lived?

```
Customer buys Product
```

No → maybe just an event.

```
Customer places Order
```

Yes → make Order an entity.

---

### 2. Does the relationship have attributes?

```
Student enrolled in Course

grade
semester
enrollment_date
```

Yes → create `Enrollment`.

---

### 3. Does the relationship have a lifecycle?

```
Pending
Approved
Completed
Cancelled
```

Yes → create an entity.

---

### 4. Does it need independent commands?

```
cancelOrder(orderId)
payInvoice(invoiceId)
closeTicket(ticketId)
```

Yes → it deserves an identity.

---

# Compression heuristic

```
A ↔ B
```

Ask:

```
Does A-B have:
    state?
    time?
    money?
    permissions?
    status?
    history?
    independent commands?
```

If yes:

```
A ← RelationshipEntity → B
```

Give it:

```
id
state machine
invariants
commands
```

Examples:

```
Vehicle + Spot      -> ParkingSession
Customer + Products -> Order
User + Device       -> Session
Student + Course    -> Enrollment
```

---

This is actually one of the most important recurring patterns in system design. In database design it is **associative entities**, in DDD it is often a **domain entity/aggregate**, in graph theory **edge reification**, and in category theory a **span**. It is the formal move of turning a **relationship into a first-class stateful object**.


## Attempt 2

### requirements:

1. Park and unpark car tracking only, no fees for now

In [ ]:

from enum import Enum
import uuid

class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

class ParkingSpot:
    def __init__(self):
        self.occupant = None
        self.type = None

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # check unpark for later.
    def release(self, vehicle):
        if self.occupant == vehicle:
            self.occupant = None
            return True
        else:
            return False
    
    def get_occupant(self):
        return self.occupant

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type = None, ticket = None):
        self.id = id
        self.type = type
        self.ticket = ticket
        

# TODO: design Parking Ticket.

class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor_id = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.floor_id = floor_id


class ParkingFloor:
    def __init__(self,size):
        self.parking_grid = {spot_id: ParkingSpot(type = 'truck' if spot_id % 2 == 0 else 'car') for spot_id in range(size//2)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                ticket = ParkingTicket(spot_id = parking_spot, vehicle_id = vehicle.id, entry_time = int(time.time()), id=uuid.uuid4())
                self.parking_grid[parking_spot].occupant.ticket = ticket
                return ticket

    def unpark(self, ticket):
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot and parking_spot.release(parking_spot.get_occupant()):
            self.filled -= 1
            del parking_spot.occupant.ticket
            return True
        return False
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size)]
        self.tickets = {}
    def park(self, vehicle):
        parked = False
        for parking_floor in self.parking_floors:
            ticket = parking_floor.park(vehicle)
            if ticket:
                parked = True
                self.tickets[ticket.id] = ticket
                break
        return parked

    def unpark(self,ticket):
        if ticket.id in self.tickets and :
            unparked = False
            for parking_floor in self.parking_floors:
                if parking_floor.unpark(self.tickets[ticket.id]):
                    unparked = True
                    del self.tickets[ticket.id]
                    break
            return unparked
        return False


### Critique After Attempt 2 (latest)

1. Findings

- High: Requirements are narrowed to "park/unpark car tracking only, no fees," but the implementation still declares Bike/Car/Truck without a clear scope contract. Lock scope explicitly (single type vs multi-type) before modeling entities.
- High: Invariants are missing as always-true statements. Right now correctness is implicit in methods, so there is no explicit guarantee for core rules like "one active ticket per parked vehicle" and "one occupied vehicle per spot." 
- High: Ownership is still unstable. `ParkingFloor` performs lifecycle mutation (`park/unpark`) while `ParkingLot` also owns global active-ticket state. This split makes invariant enforcement ambiguous.
- High: The state machine is not explicit. Legal/illegal transitions (e.g., unpark with stale/forged ticket, double unpark, park when full) are not modeled first, so behavior is ad hoc in method branches.
- High: Core model has execution-breaking mismatches: `ParkingSpot.__init__` takes no `type` but is called with `type=...`; `time` is used but not imported; `ParkingTicket` constructor is called with unsupported `id` argument; `ParkingFloor(size)` creates only `size//2` spots but `full()` compares against `size`.
- Medium: Ticket identity flow is inconsistent. `ParkingLot.unpark` accepts a ticket object, but true authority should be `ticket_id -> active session` lookup; current API makes forged object usage easier and weakens boundary checks.
- Medium: `Vehicle.ticket` as mutable attached state is not clearly owned. During unpark, ticket cleanup mutates occupant internals directly (`del ...ticket`), which couples spot release with vehicle mutation authority.
- Medium: DS/operations are only partially aligned. Linear scan is fine for now, but "nearest available spot" is a requirement and is not represented in structure or ordering policy.
- Medium: Happy/failure traces are missing for this latest attempt revision, so ownership and transition gaps are not validated end-to-end.

2. Revision order

1. Rewrite section 1 with explicit scope contract for this iteration (single floor? multi-floor? which vehicle types? no fee yet).
2. Add 3 to 5 invariants as always-true statements, each with an owner that enforces it.
3. Write the spot/session state machine with legal and illegal transitions before class edits.
4. Rewrite responsibilities in `Rule -> Owner -> Mutator -> Enforcement` form, especially ticket lifecycle and unpark authority.
5. Refactor API boundary to `unpark(ticket_id)` and centralize active-session validation in `ParkingLot`.
6. Re-run one happy path and one failure path trace, then patch code to match those traces.


## 5 questions to Challenge my understanding


  1. Which single object is the mutation authority for transitioning a spot from empty to
     occupied, and what invariant would fail first if another object also mutates it?

   ans. The mutation authority should be owned by ParkingLot itself, but state tracking should be done by ParkingSpot. The Invariant of occupation, track of local and hence global ParkingLot would be violated

  2. If unpark receives a forged or stale ticket id, what exact enforcement point rejects
     it, and what state remains unchanged after rejection?
     ans. the parking lot ticket hashmap should enforce valid ticket ids. The occupancy should remain unchanged after rejection.

  3. What are your legal and illegal transitions for one parking session from creation to
     close, including the double-unpark case?
     ans. Legal: if ticket is valid then unpark, if there is no ticket or another ticket shares same parking spot or vehicle id then illegal.

  4. For your current DS choices, what is the time complexity of park and unpark, and which
     requirement would force you to change the structure first?
      ans. Park is O(ParkingFloor.size \times ParkingLot.parking_floors)
      If the parking is ordered within the floor that the vehicle prefers to park some ordered spot, then I'd have to change the structure from dictionary within the grid to list or heap + hashmap where hashmap stores valid while, heap stores remaining spaces in priority order.

  5. If the requirement changes to “support nearest-spot allocation across floors,” which
     class boundaries stay stable and which ownership rules must move?
     ans. I don't get what class boundaries mean, however, I'm thinking ParkingLot handles the allocation logic including parking ticket creation. while parkingfloor

### Critique of your 5-question responses and guidance before Attempt 3

1. Findings

- High: You are still mixing **state owner** with **mutation authority**. Saying `ParkingLot` owns the mutation while `ParkingSpot` tracks the state is only safe if you state the rule precisely: `ParkingSpot` owns occupancy state, while `ParkingLot` orchestrates transitions and validates system-level invariants before calling into the spot. Without that split, ownership is still ambiguous.
- High: Your answer to the stale/forged ticket question is directionally correct but incomplete. The rejection point is not just "the hashmap"; it is the `ParkingLot.unpark(ticket_id)` validation boundary that checks `ticket_id -> active session` before any floor or spot mutation. You must name what does **not** change: ticket map, spot occupancy, and any vehicle-session association.
- High: Your lifecycle answer is still under-modeled. `valid ticket then unpark` is not a state machine. You need explicit session states and transitions, for example `NONE -> ACTIVE -> CLOSED`, plus illegal operations like `CLOSED -> CLOSED` via double unpark and `NONE -> CLOSED` via forged ticket.
- Medium: Your DS answer shows decent instincts, but you skipped `unpark` complexity and the current authoritative lookup shape. With `ticket_id -> active ticket`, unpark should target O(1) session lookup plus direct floor/spot resolution from the ticket. If you need to scan floors on unpark, your identity model is still weak.
- Medium: You identified `ParkingLot` as the right place for allocation logic, but you do not yet have a clean statement of what stays stable under the nearest-spot change. The stable boundary is roughly: `ParkingLot` remains the orchestration root, `ParkingFloor` remains local spot inventory, `ParkingSpot` remains occupancy state holder. The moving part is allocation policy and any data structure supporting ordered selection.
- Medium: You are still answering with implementation fragments before locking the drill artifacts. The earlier critique asked for requirements, invariants, state machine, ownership table, and traces first. Attempt 3 should not jump back into code until those are explicit.

2. More guidance

1. Rewrite ownership with one sentence per mutable state: `spot occupancy is owned by ParkingSpot`; `active parking sessions are owned by ParkingLot`; `floor inventory is owned by ParkingFloor`.
   Formalization: this split follows the rule that each mutable part of the abstract state should have one primary state owner in `X`, while orchestration may live elsewhere. `ParkingSpot` owns occupancy because occupancy is a local spot-state predicate (`empty` or `occupied(vehicle)`), and keeping that field on the spot makes the occupancy invariant checkable at the state carrier. `ParkingLot` owns active parking sessions because the session index (`ticket_id -> active session`) is a system-level representation that crosses floors and is the guard point for legal `unpark(ticket_id)` transitions. `ParkingFloor` owns floor inventory because membership of spots to a floor is floor-local structure, and nearest-spot or compatibility scans depend on that local collection boundary. In formal terms, the ownership map should minimize diffuse mutation authority: local state stays with the smallest stable carrier, while cross-cutting indexes and transition guards stay at the aggregate root.
2. Write invariants as always-true claims, not workflow steps. Good examples here are: `an occupied spot has exactly one active ticket`, `an active ticket refers to exactly one occupied spot`, `a closed ticket cannot be used to unpark again`.
3. Write the parking-session state machine separately from the spot state machine. You are currently collapsing them into one vague notion of "valid ticket". For the current simplified requirements, you do not need a large lifecycle, but you still need at least the conceptual distinction between `no session`, `active session`, and `closed/nonexistent for active use`. Otherwise you cannot state clearly what a stale ticket means or why double-unpark is illegal. If you want a minimal model, it is acceptable to represent this as `ticket exists in active_sessions` vs `ticket does not exist in active_sessions`, which is effectively a 2-state session model. That means yes: for this round, you can avoid storing historical closed tickets if fees, receipts, audits, and reporting are out of scope. But the design must still model the transition from active to inactive explicitly. In many real business systems, tickets are retained after close for billing disputes, audit logs, receipts, analytics, and lost-ticket handling, which is why richer ticket states are common even if your current exercise does not require them.
4. Make `unpark(ticket_id)` the public boundary for Attempt 3. Let `ParkingLot` validate the active session, then delegate the physical release to the floor/spot level.
5. Run one explicit happy path and one failure path in text before coding: `park(car)` success, and `unpark(stale_ticket_id)` rejection. This will force you to say whether a stale ticket means `closed ticket still stored` or `ticket no longer present in active_sessions`; both are valid models, but you need to choose one deliberately.
6. Treat requirement changes as a boundary test: ask which existing owner keeps its job and which policy or DS must change. That is what `class boundaries stay stable` means in this context.

3. Gap matrix

| Area | What you currently understand | Current gap | What to fix in Attempt 3 |
| --- | --- | --- | --- |
| Requirements | You know the scope must be narrowed per iteration. | Your scope is still half-specified and drifts between simplified and full problem requirements. | State exact scope in 3 to 5 bullets before design. |
| Invariants | You recognize occupancy consistency matters. | You describe consequences loosely instead of writing enforceable always-true statements. | Write 3 to 5 invariants with a named owner for each. |
| State machine | You know validity of ticket matters. | You have not enumerated states, legal transitions, or illegal transitions. | Write session and spot transitions explicitly. |
| Ownership | You sense `ParkingLot` should orchestrate more. | You still blur owner, mutator, and enforcer. | Fill `Rule -> Owner -> Mutator -> Enforcement` for park and unpark. |
| Identity / API boundary | You understand ticket lookup is important. | You still phrase unpark around a vague ticket object or validity notion. | Use `ticket_id` as the external command input and map it to an active session. |
| Data structures | You see ordered allocation may require better DS. | You did not connect DS choice to current operations, especially unpark. | State `park` and `unpark` target complexity and choose DS to support them. |
| Extensibility | You correctly place nearest-spot logic near lot-level orchestration. | You do not yet separate stable boundaries from changing policy. | Keep owners stable; move only allocation strategy and supporting indexes. |
| Trace discipline | You can reason locally about method behavior. | You are still not validating the whole design through one happy and one failure path. | Write both traces before touching code. |




## TODOs before Attempt 3 code

1. Write a 3 to 5 bullet scope section.
   Why: your requirements are still drifting between a simplified exercise and the full parking-lot problem, which makes the rest of the design unstable.
   1. Park(vehicle), unpark(ticket) is handled by ParkingLot. No fee calculation for now, but entry and exit time is ok, (perhaps this is shown on the parking ticket upon exit)
   2. ParkingLot is the main entry point for all operations
   3. unpark should return a fee if applicable
2. Write 3 to 5 invariants with an explicit owner for each.
   Why: the critique shows you understand consequences of bad state, but not yet the always-true rules that the design must preserve.
   1. ParkingLot.tickets only tracks valid tickets
   2. (ParkingFloor[i][ParkingSpot]) always tracks occupancy to the exact time in coordination with parkingTicket of the vehicle within 1 to 1 mapping  of active tickets.
   
3. Write a minimal session state model and a spot state model.
   Why: you still describe validity informally, so stale ticket handling and double-unpark rejection are not grounded in explicit states.
   ParkingLot.tickets will store active and validated tickets for audit history, park creates ticket which is active, and unpark validates the ticket and marks it as inactive.

4. Fill a `Rule -> Owner -> Mutator -> Enforcement` table for `park` and `unpark`.
   Why: your biggest conceptual gap is still mixing state ownership, mutation authority, and orchestration.
   ans. I don't get this question. but the ticketstatus within the self.tickets active inactive and leaving unpark is inactive if precondition was active is enforced in ParkingLot. And I also made parkinglot own the ParkingTicket creation instead of Parkingfloor 

5. Choose the external unpark API as `unpark(ticket_id)` and state what `active_sessions` stores.
   Why: your identity boundary is still underspecified, and that boundary determines both legality checks and unpark complexity.

   ans. i decided to use self.tickets which stores full audit history of tickets, activity is by ticket.status. legality check is O(1) by hashmap in this case

6. State target time complexity for `park` and `unpark`, then justify the DS.
   Why: your DS answer had the right instinct but was not yet tied to the actual operations and current representation.

   ans. park is O(N x M) where N is the number of floors and M is the number of spots per floor, unpark is O(1) by hashmap

7. Write one happy path and one failure path in plain text.
   Why: the critique keeps finding missing end-to-end validation, and traces are the fastest way to expose ownership mistakes before coding.

   ans. I'm not sure if the vehicle should store the ticket or not in the realworld, of course the ticket is stored by global parkinglot and stores the vehicle for validation invalidation, since the controller is from parkinglot. for simplification since we're doing parkinglot instead of vehicle control side implemenation, I'm making vehicle not store the ticket

8. Only after the above, revise the code cell to match the written model.
   Why: your current pattern is still code-first, while the earlier critique and this one both show the model is not stable enough yet.

## Attempt 3 (more guidance)

3. explicit scope contract: ParkingLot manages multiple floors, and for the moment cars and trucks of the specific configuration half half. later what should it be?
a. ParkingSpot owns spot occupancy, ParkingLot owns active parking sessions, floor inventory owned by parking floor. 

4. occupied spot <-> active ticket 1 to 1, closed ticket can't be used to unpark again




In [ ]:

from enum import Enum
import uuid


class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type: VehicleType = None, ticket = None):
        self.id = id
        self.type = type


class ParkingSpot:
    def __init__(self, type: VehicleType):
        self.occupant = None
        self.type = type

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # check unpark for later. #give the vehicle
    def release(self, vehicle_id) -> Vehicle:
        if self.occupant and self.occupant.id == vehicle_id:
            ret = self.occupant
            self.occupant = None
            return ret
        else:
            return None
    
    def get_occupant(self):
        return self.occupant


class TicketStatus(Enum):
    Active = "Active"
    Inactive = "Inactive"    


class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor: int = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.exit_time = entry_time
        self.floor = floor
        self.status = TicketStatus.Active


class ParkingFloor:
    def __init__(self,size):

        self.parking_grid = {spot_id: ParkingSpot(type = 'truck' if spot_id % 2 == 0 else 'car') for spot_id in range(size//2)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    #shifted ownership of ticket creation to parkinglot too. but why?
    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                return spot_id

    def unpark(self, ticket) -> Vehicle:
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot:
            vehicle = parking_spot.release(ticket.vehicle_id)
            if vehicle:
                self.filled -= 1
                return vehicle
        return None
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size) for _ in range(parking_floors)]
        self.tickets = {} #tracks all active and inactive tickets tickets
        self.time = 0
        self.total_earnings = 0

    def park(self, vehicle):
        ticket = None
        for idx, parking_floor in enumerate(self.parking_floors):
            spot_id = parking_floor.park(vehicle)
            if spot_id:
                ticket = ParkingTicket(spot_id=spot_id, vehicle_id=vehicle.id, entry_time=self.time, floor=idx)
                self.tickets[ticket.id] = ticket
                break
        return ticket

    def calculate_fees(self, ticket):
        # direct 1 per hour cost
        return ticket.exit_time - ticket.entry_time 

    def time_increment(self):
        self.time += 1 
        
    def unpark(self,ticket):
        if ticket.floor < len(self.parking_floors) and ticket.id in self.tickets and self.tickets[ticket.id].status == TicketStatus.Active:
            vehicle = self.parking_floors[ticket.floor].unpark(self.tickets[ticket.id])
            if vehicle:
                vehicle.ticket.exit_time = self.time
                ticket.fees = self.calculate_fees(vehicle.ticket)
                self.total_earnings += ticket.fees
                ticket.status = TicketStatus.Inactive
                return ticket.fees
        return None


### Critique After Attempt 3

1. Findings

- High: your current code still violates the narrowed model in execution-critical ways. `ParkingFloor.parking_grid` builds spot types as raw strings (`'truck'`, `'car'`) while `Vehicle.type` is a `VehicleType` enum, so `ParkingSpot.assign()` will reject every vehicle.
- High: `ParkingLot.park()` treats `spot_id` as truthy, so a valid spot at index `0` is treated as failure. This is a state transition bug, not a syntax issue.
- High: you intentionally stopped storing `vehicle.ticket`, but `ParkingLot.unpark()` still writes through `vehicle.ticket.exit_time`. That means your ownership direction improved, but the code path still depends on the old model and will fail.
- High: your floor capacity model is still inconsistent. `ParkingFloor` creates `size//2` spots but `full()` compares `filled == size`, so reachability and capacity invariants cannot hold.
- High: the external unpark boundary is still wrong for the design you said you want. The code uses `unpark(ticket)` instead of `unpark(ticket_id)`, which keeps identity and legality checks weaker than your own Attempt 3 notes.
- Medium: progression is real and positive. Moving ticket creation to `ParkingLot`, adding `TicketStatus`, and making the ticket carry `floor` are all better ownership signals than Attempt 2. Those are structural improvements, not cosmetic ones.
- Medium: your requirement bullets are still unstable. You say "no fee calculation for now" and also "unpark should return a fee if applicable". That ambiguity leaks directly into `calculate_fees`, `exit_time`, and `total_earnings`.
- Medium: your invariants are closer, but they are still partly workflow descriptions. `ParkingLot.tickets only tracks valid tickets` conflicts with your later statement that the map stores audit history with active and inactive tickets. Pick one representation and state it precisely.
- Medium: your inline comment intuition is mixed. `# shifted ownership of ticket creation to parkinglot too. but why?` is asking the right design question; the answer is that ticket issuance is a cross-floor session creation step guarded by lot-level invariants. In contrast, `#give the vehicle` on `release()` is not a design insight; it hides the more important question, which is whether `release` should consume `vehicle_id`, `ticket`, or no external identity at all.
- Medium: you still have not written the happy path and failure path as traces before coding, and that is why old-model and new-model assumptions are now mixed inside the same methods.

2. Progression critique

- Stronger than Attempt 2: you moved session creation upward to `ParkingLot`, introduced explicit active/inactive ticket status, and started separating session ownership from spot occupancy.
- Still not stable: you changed some code to the new ownership model, but not all dependent paths. That means the design is improving faster than the implementation discipline.
- Main pattern in your progression: your intuition about ownership is getting better, but you still convert that intuition into code before the full transition model is written down.

3. In-code comment intuition check

| Location | Intuition quality | Critique |
| --- | --- | --- |
| `# TODO: design vehicle` | Medium | Reasonable placeholder, but the real question is whether `Vehicle` is only identity plus type for this problem or whether it also owns parking-session state. Your later code suggests it should not own the ticket. |
| `# contracts: true if operation worked as expected, else false.` | Medium | The intuition about method contracts is fine, but it is underspecified. The design needs preconditions and invariant preservation, not just success booleans. |
| `# check unpark for later. #give the vehicle` | Low | This comment does not identify the important design choice. The core question is who authorizes release and what identity the release consumes. |
| `# shifted ownership of ticket creation to parkinglot too. but why?` | High | Good intuition. Ticket issuance belongs with the aggregate root because it creates the cross-floor active session index and enforces lot-level uniqueness and legality. |
| `self.tickets = {} #tracks all active and inactive tickets tickets` | Medium | The comment identifies the intended representation, but it conflicts with your earlier invariant wording about only valid tickets. Tighten the invariant to match the representation. |
| `# direct 1 per hour cost` | Low | This is implementation-first while requirements are still unstable. Until fees are clearly in scope, this comment introduces accidental product behavior. |

4. Gap matrix

| Area | Progress since Attempt 2 | Remaining gap | What to change next |
| --- | --- | --- | --- |
| Requirements | You are trying to narrow scope before coding. | Fee support is still contradictory. | Decide now: fees in scope or out of scope for Attempt 3. |
| Invariants | You started expressing 1-to-1 ticket/spot thinking. | Invariants still conflict with the chosen ticket-history representation. | Rewrite invariants against the actual data model. |
| State machine | You introduced `TicketStatus`. | Spot lifecycle and stale-ticket handling are still not written as explicit transitions. | Write `spot` and `ticket/session` state machines before more code changes. |
| Ownership | You moved ticket creation to `ParkingLot`. | Old ownership assumptions still survive in `unpark()`. | Remove `vehicle.ticket` dependence and centralize validation in lot-level logic. |
| API boundary | You understand ticket identity matters more. | Public API still accepts a ticket object. | Change to `unpark(ticket_id)` and keep ticket lookup internal. |
| Data representation | You introduced `status` and `floor` on tickets. | Representation still has type mismatch and capacity mismatch bugs. | Make `VehicleType` consistent end-to-end and fix floor capacity math. |
| Trace discipline | You are asking better design questions in comments. | You still have no explicit happy/failure traces. | Write both traces and then align code line-by-line. |

5. Revision order

1. Resolve the scope contradiction around fees. If fees are out, remove `calculate_fees`, `total_earnings`, and `exit_time` handling for now.
2. Rewrite invariants to match one chosen ticket model: either `tickets stores active only` or `tickets stores all sessions and status distinguishes active vs inactive`.
3. Write the minimal state machines explicitly: `spot: EMPTY -> OCCUPIED -> EMPTY`; `session: ABSENT -> ACTIVE -> INACTIVE` or `ticket_id not in map` if you choose deletion instead of retention.
4. Change the external command to `unpark(ticket_id)` and make `ParkingLot` do `ticket_id -> session -> floor/spot` resolution.
5. Fix the representation mismatches before any further design judgment: enum-vs-string spot types, `spot_id == 0` handling, floor capacity math, and `vehicle.ticket` removal.
6. Add one happy path and one failure path trace in text, then check each line of code against the trace.

6. Challenge questions

1. If `tickets` stores both active and inactive sessions, what exact invariant distinguishes a reusable stale id from an auditable closed session?
2. Where is the single enforcement point that prevents a ticket from being closed twice, and what state is unchanged when that enforcement rejects the second close?
3. If you remove `vehicle.ticket` entirely, which object now carries enough information to compute exit behavior and why is that ownership cleaner?
4. Under your current representation, what must be true about `ticket.floor` and `ticket.spot_id` for `unpark(ticket_id)` to avoid scanning unrelated floors or spots?
5. If fees are out of scope today but added tomorrow, which part of `delta` changes and which invariants should stay unchanged?

7. Optional deeper model

Formalization of your current design direction:

- `Sigma` (abstract state): spot occupancy across floors, active/inactive parking-session status, current lot time, and optionally earnings if fees are in scope.
- `X` (representation): `parking_floors`, each floor's `parking_grid`, and `tickets` as the session index. If you keep audit history, `tickets` represents all sessions plus status; if not, it represents active sessions only.
- `I` (invariants): `each occupied spot corresponds to exactly one active session`; `each active session refers to exactly one occupied spot`; `an inactive session cannot trigger another unpark`; `filled` equals the number of occupied spots on a floor`.
- `delta` (transitions): `park(vehicle)` should create a legal `EMPTY -> OCCUPIED` spot transition and `ABSENT -> ACTIVE` session transition together; `unpark(ticket_id)` should create `OCCUPIED -> EMPTY` and `ACTIVE -> INACTIVE` together or reject without mutation.
- `Phi` (guards): spot compatibility, capacity, ticket existence, ticket active status, and floor/spot reference validity.
- `A` (mutation authority sketch): `ParkingSpot` owns occupancy field mutation; `ParkingFloor` owns local spot inventory lookup; `ParkingLot` owns session creation/closure authority and legality checks for external commands.

The key formal mistake still present in code is transition non-preservation: the new ownership model in your notes is not yet preserved by the actual `unpark()` implementation, because the code still depends on `vehicle.ticket` and a ticket-object boundary.


In [ ]:
from enum import Enum
import uuid


class VehicleType(Enum):
    Bike = "Bike"
    Car = "Car"
    Truck = "Truck"

# TODO: design vehicle
class Vehicle:
    def __init__(self, id = None, type: VehicleType = None, ticket = None):
        self.id = id
        self.type = type


class ParkingSpot:
    def __init__(self, type: VehicleType):
        self.occupant = None
        self.type = type

    # contracts: true if operation worked as expected, else false.
    def assign(self, vehicle):
        if self.occupant is None and vehicle.type == self.type:
            self.occupant = vehicle
            return True
        else:
            return False
    
    # check unpark for later. #give the vehicle
    def release(self, vehicle_id) -> Vehicle:
        if self.occupant and self.occupant.id == vehicle_id:
            ret = self.occupant
            self.occupant = None
            return ret
        else:
            return None
    
    def get_occupant(self):
        return self.occupant


class TicketStatus(Enum):
    Active = "Active"
    Inactive = "Inactive"    


class ParkingTicket:
    def __init__(self, spot_id = None, vehicle_id = None, entry_time = None, floor: int = None):
        self.id = str(uuid.uuid4())
        self.spot_id = spot_id
        self.vehicle_id = vehicle_id
        self.entry_time = entry_time
        self.exit_time = entry_time
        self.floor = floor
        self.status = TicketStatus.Active


class ParkingFloor:
    def __init__(self,size):

        self.parking_grid = {spot_id: ParkingSpot(type = VehicleType.Bike if spot_id % 2 == 0 else VehicleType.Car) for spot_id in range(size//2)}
        self.size = size
        self.filled = 0

    def full(self):
        return self.filled == self.size

    #shifted ownership of ticket creation to parkinglot too. but why?
    def park(self, vehicle):
        if not self.full():
            parking_spot = None
            for spot_id, spot in self.parking_grid.items():
                if spot.assign(vehicle):
                    parking_spot = spot_id
                    break
            if parking_spot is not None:
                self.filled += 1
                return spot_id

    def unpark(self, ticket) -> Vehicle:
        parking_spot = self.parking_grid.get(ticket.spot_id)
        if parking_spot:
            vehicle = parking_spot.release(ticket.vehicle_id)
            if vehicle:
                self.filled -= 1
                return vehicle
        return None
    

class ParkingLot:
    def __init__(self, parking_floors = 10, parking_floor_size = 10):
        self.parking_floors = [ParkingFloor(size = parking_floor_size) for _ in range(parking_floors)]
        self.tickets = {} #tracks all active and inactive tickets tickets
        self.time = 0
        self.total_earnings = 0

    def park(self, vehicle):
        ticket = None
        for idx, parking_floor in enumerate(self.parking_floors):
            spot_id = parking_floor.park(vehicle)
            if spot_id:
                ticket = ParkingTicket(spot_id=spot_id, vehicle_id=vehicle.id, entry_time=self.time, floor=idx)
                self.tickets[ticket.id] = ticket
                break
        return ticket

    def calculate_fees(self, ticket):
        # direct 1 per hour cost
        return ticket.exit_time - ticket.entry_time 

    def time_increment(self):
        self.time += 1 
        
    def unpark(self,ticket):
        if ticket.floor < len(self.parking_floors) and ticket.id in self.tickets and self.tickets[ticket.id].status == TicketStatus.Active:
            vehicle = self.parking_floors[ticket.floor].unpark(self.tickets[ticket.id])
            if vehicle:
                vehicle.ticket.exit_time = self.time
                ticket.fees = self.calculate_fees(vehicle.ticket)
                self.total_earnings += ticket.fees
                ticket.status = TicketStatus.Inactive
                return ticket.fees
        return None
